In [1]:

import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
from skimage.metrics import structural_similarity, peak_signal_noise_ratio

# --------------------------------------------------
# 1. PROJECT CONFIGURATION
# --------------------------------------------------

ROOT = Path(r"D:\PS07_CT2MRI")

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "dataset"))

from multi_patient_loader import (
    MultiPatientDataset,
    PatientBatchSampler
)

from resunet import ResidualUNet
from patchgan_discriminator import PatchGANDiscriminator
from normalise import normalise_ct, normalise_mri

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 1
EPOCHS = 10
LEARNING_RATE = 2e-4
LAMBDA_L1 = 100
VAL_BATCH_SIZE = 8
SEED = 42

PIX2PIX_DIR = ROOT / "models" / "pix2pix"
OUTPUT_DIR = ROOT / "outputs" / "pix2pix"
RESULTS_DIR = ROOT / "results" / "pix2pix"

for folder in [PIX2PIX_DIR, OUTPUT_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. REPRODUCIBILITY
# --------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# --------------------------------------------------
# 3. INITIALIZE DATASETS
# --------------------------------------------------

print("Device:", DEVICE)

train_dataset = MultiPatientDataset(
    ROOT / "train_patients.csv"
)

val_dataset = MultiPatientDataset(
    ROOT / "val_patients.csv"
)

train_sampler = PatientBatchSampler(
    train_dataset,
    BATCH_SIZE,
    shuffle_patients=True
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    num_workers=0
)

print("\nTraining patients:", len(train_dataset.patient_ids))
print("Training samples:", len(train_dataset))
print("Training batches:", len(train_loader))

print("Validation patients:", len(val_dataset.patient_ids))

# --------------------------------------------------
# 4. INITIALIZE GENERATOR AND DISCRIMINATOR
# --------------------------------------------------

generator = ResidualUNet(
    in_channels=3,
    out_channels=1,
    base_channels=16
).to(DEVICE)

discriminator = PatchGANDiscriminator(
    in_channels=4,
    base_channels=64
).to(DEVICE)

print("\nGenerator parameters:",
      sum(p.numel() for p in generator.parameters()))

print("Discriminator parameters:",
      sum(p.numel() for p in discriminator.parameters()))

# --------------------------------------------------
# 5. LOSS FUNCTIONS AND OPTIMIZERS
# --------------------------------------------------

gan_criterion = nn.BCEWithLogitsLoss()
l1_criterion = nn.L1Loss()

optimizer_G = torch.optim.Adam(
    generator.parameters(),
    lr=LEARNING_RATE,
    betas=(0.5, 0.999)
)

optimizer_D = torch.optim.Adam(
    discriminator.parameters(),
    lr=LEARNING_RATE,
    betas=(0.5, 0.999)
)

scaler = GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

# --------------------------------------------------
# 6. FULL-VOLUME VALIDATION
# --------------------------------------------------

@torch.no_grad()
def validate_patient(patient_id):

    folder = ROOT / "dataset" / "volumes" / patient_id

    ct_img = nib.load(str(folder / "ct.nii.gz"))
    mr_img = nib.load(str(folder / "mr.nii.gz"))
    mask_img = nib.load(str(folder / "mask.nii.gz"))

    ct = np.asarray(ct_img.dataobj, dtype=np.float32)
    mr = np.asarray(mr_img.dataobj, dtype=np.float32)
    mask = np.asarray(mask_img.dataobj, dtype=np.uint8)

    if not (ct.shape == mr.shape == mask.shape):
        raise ValueError(f"Shape mismatch: {patient_id}")

    ct = normalise_ct(ct)
    mr = normalise_mri(mr, mask)

    depth = ct.shape[2]

    prediction = np.zeros(
        ct.shape,
        dtype=np.float32
    )

    generator.eval()

    for start in range(0, depth, VAL_BATCH_SIZE):

        end = min(start + VAL_BATCH_SIZE, depth)

        batch_slices = []

        for z in range(start, end):

            previous_z = max(z - 1, 0)
            next_z = min(z + 1, depth - 1)

            ct_stack = np.stack([
                ct[:, :, previous_z],
                ct[:, :, z],
                ct[:, :, next_z]
            ], axis=0)

            batch_slices.append(ct_stack)

        batch = torch.from_numpy(
            np.stack(batch_slices).astype(np.float32)
        ).to(DEVICE)

        with autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):

            output = generator(batch)

        output = output.float().cpu().numpy()

        for i, z in enumerate(range(start, end)):
            prediction[:, :, z] = output[i, 0]

    prediction = np.clip(prediction, 0, 1)

    # Full-volume MAE
    full_mae = float(
        np.mean(np.abs(prediction - mr))
    )

    # Supplied-mask MAE
    mask_bool = mask > 0

    if mask_bool.any():
        masked_mae = float(
            np.mean(
                np.abs(prediction[mask_bool] - mr[mask_bool])
            )
        )
    else:
        masked_mae = np.nan

    # Mean 2D slice SSIM
    ssim_values = []

    for z in range(depth):

        score = structural_similarity(
            mr[:, :, z],
            prediction[:, :, z],
            data_range=1.0
        )

        ssim_values.append(score)

    mean_ssim = float(np.mean(ssim_values))

    # Full-volume PSNR
    psnr = float(
        peak_signal_noise_ratio(
            mr,
            prediction,
            data_range=1.0
        )
    )

    # Save predicted MRI
    output_img = nib.Nifti1Image(
        prediction.astype(np.float32),
        ct_img.affine,
        ct_img.header
    )

    output_img.set_data_dtype(np.float32)

    nib.save(
        output_img,
        str(OUTPUT_DIR / f"{patient_id}_predicted_mri.nii.gz")
    )

    return {
        "patient_id": patient_id,
        "full_volume_mae": full_mae,
        "masked_mae": masked_mae,
        "mean_slice_ssim": mean_ssim,
        "full_volume_psnr": psnr
    }

# --------------------------------------------------
# 7. VALIDATE ALL FIVE PATIENTS
# --------------------------------------------------

def run_validation():

    generator.eval()

    results = []

    for patient_id in val_dataset.patient_ids:

        print("\nValidating:", patient_id)

        metrics = validate_patient(patient_id)

        results.append(metrics)

        print(
            f"Full MAE: {metrics['full_volume_mae']:.6f} | "
            f"Masked MAE: {metrics['masked_mae']:.6f} | "
            f"SSIM: {metrics['mean_slice_ssim']:.6f} | "
            f"PSNR: {metrics['full_volume_psnr']:.4f}"
        )

    df = pd.DataFrame(results)

    return df

# --------------------------------------------------
# 8. TRAINING LOOP
# --------------------------------------------------

history = []
best_masked_mae = float("inf")

for epoch in range(EPOCHS):

    generator.train()
    discriminator.train()

    running_G = 0.0
    running_D = 0.0
    running_L1 = 0.0
    running_GAN = 0.0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for ct, real_mri, mask in progress:

        ct = ct.to(DEVICE)
        real_mri = real_mri.to(DEVICE)

        # ------------------------------------------
        # GENERATOR FORWARD
        # ------------------------------------------

        optimizer_G.zero_grad(set_to_none=True)
        optimizer_D.zero_grad(set_to_none=True)

        with autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):

            fake_mri = generator(ct)

        # ------------------------------------------
        # DISCRIMINATOR UPDATE
        # ------------------------------------------

        optimizer_D.zero_grad(set_to_none=True)

        with autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):

            real_logits = discriminator(
                ct, real_mri
            )

            fake_logits = discriminator(
                ct, fake_mri.detach()
            )

            real_labels = torch.ones_like(real_logits)
            fake_labels = torch.zeros_like(fake_logits)

            loss_D_real = gan_criterion(
                real_logits, real_labels
            )

            loss_D_fake = gan_criterion(
                fake_logits, fake_labels
            )

            loss_D = 0.5 * (
                loss_D_real + loss_D_fake
            )

        scaler.scale(loss_D).backward()
        scaler.step(optimizer_D)

        # ------------------------------------------
        # GENERATOR UPDATE
        # ------------------------------------------

        for parameter in discriminator.parameters():
            parameter.requires_grad_(False)

        optimizer_G.zero_grad(set_to_none=True)

        with autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):

            generated_logits = discriminator(
                ct, fake_mri
            )

            target_labels = torch.ones_like(
                generated_logits
            )

            loss_G_GAN = gan_criterion(
                generated_logits, target_labels
            )

            loss_G_L1 = l1_criterion(
                fake_mri, real_mri
            )

            loss_G = (
                loss_G_GAN +
                LAMBDA_L1 * loss_G_L1
            )

        scaler.scale(loss_G).backward()
        scaler.step(optimizer_G)

        for parameter in discriminator.parameters():
            parameter.requires_grad_(True)

        scaler.update()

        running_G += loss_G.item()
        running_D += loss_D.item()
        running_L1 += loss_G_L1.item()
        running_GAN += loss_G_GAN.item()

        progress.set_postfix(
            G=f"{loss_G.item():.4f}",
            D=f"{loss_D.item():.4f}",
            L1=f"{loss_G_L1.item():.4f}"
        )

    # ----------------------------------------------
    # EPOCH TRAINING METRICS
    # ----------------------------------------------

    train_G = running_G / len(train_loader)
    train_D = running_D / len(train_loader)
    train_L1 = running_L1 / len(train_loader)
    train_GAN = running_GAN / len(train_loader)

    print(f"\nEpoch {epoch + 1} completed")
    print("Generator loss:", train_G)
    print("Discriminator loss:", train_D)
    print("L1 loss:", train_L1)
    print("Adversarial loss:", train_GAN)

    # ----------------------------------------------
    # VALIDATION
    # ----------------------------------------------

    print("\nRunning full-volume validation...")

    validation_df = run_validation()

    validation_df.to_csv(
        RESULTS_DIR / f"validation_epoch_{epoch + 1}.csv",
        index=False
    )

    avg_metrics = validation_df.drop(
        columns=["patient_id"]
    ).mean()

    val_masked_mae = float(
        avg_metrics["masked_mae"]
    )

    val_full_mae = float(
        avg_metrics["full_volume_mae"]
    )

    val_ssim = float(
        avg_metrics["mean_slice_ssim"]
    )

    val_psnr = float(
        avg_metrics["full_volume_psnr"]
    )

    print("\nValidation averages:")
    print(avg_metrics)

    # ----------------------------------------------
    # HISTORY
    # ----------------------------------------------

    history.append({
        "epoch": epoch + 1,
        "train_G_loss": train_G,
        "train_D_loss": train_D,
        "train_L1_loss": train_L1,
        "train_GAN_loss": train_GAN,
        "val_masked_mae": val_masked_mae,
        "val_full_mae": val_full_mae,
        "val_ssim": val_ssim,
        "val_psnr": val_psnr
    })

    pd.DataFrame(history).to_csv(
        RESULTS_DIR / "training_history.csv",
        index=False
    )

    # ----------------------------------------------
    # LATEST CHECKPOINT
    # ----------------------------------------------

    checkpoint = {
        "epoch": epoch + 1,
        "generator_state_dict": generator.state_dict(),
        "discriminator_state_dict": discriminator.state_dict(),
        "optimizer_G_state_dict": optimizer_G.state_dict(),
        "optimizer_D_state_dict": optimizer_D.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "train_G_loss": train_G,
        "train_D_loss": train_D,
        "val_masked_mae": val_masked_mae
    }

    torch.save(
        checkpoint,
        PIX2PIX_DIR / "latest_checkpoint.pth"
    )

    # ----------------------------------------------
    # BEST GENERATOR CHECKPOINT
    # ----------------------------------------------

    if np.isfinite(val_masked_mae) and val_masked_mae < best_masked_mae:

        best_masked_mae = val_masked_mae

        torch.save(
            {
                "epoch": epoch + 1,
                "generator_state_dict": generator.state_dict(),
                "val_masked_mae": val_masked_mae,
                "val_full_mae": val_full_mae,
                "val_ssim": val_ssim,
                "val_psnr": val_psnr
            },
            PIX2PIX_DIR / "best_generator.pth"
        )

        print("\nNew best generator checkpoint saved!")

    print("\nEpoch", epoch + 1, "finished.")

# --------------------------------------------------
# 9. FINAL SUMMARY
# --------------------------------------------------

print("\n====================================")
print("PIX2PIX TRAINING COMPLETED")
print("====================================")

print("Best validation masked MAE:", best_masked_mae)
print("Latest checkpoint:", PIX2PIX_DIR)
print("Training history:", RESULTS_DIR / "training_history.csv")


Device: cuda

Preparing patient slice indexes...
P1046: 179 masked + 4 background = 183 selected slices
P1068: 182 masked + 3 background = 185 selected slices
P1127: 176 masked + 5 background = 181 selected slices
P1316: 182 masked + 3 background = 185 selected slices
P1375: 178 masked + 4 background = 182 selected slices
P1389: 181 masked + 3 background = 184 selected slices
P1815: 176 masked + 5 background = 181 selected slices
P1965: 179 masked + 4 background = 183 selected slices
P2017: 181 masked + 3 background = 184 selected slices
P2052: 181 masked + 3 background = 184 selected slices
P2161: 180 masked + 4 background = 184 selected slices
P2341: 180 masked + 4 background = 184 selected slices
P2349: 176 masked + 5 background = 181 selected slices
P2374: 181 masked + 3 background = 184 selected slices
P2420: 180 masked + 4 background = 184 selected slices
P2622: 181 masked + 3 background = 184 selected slices
P2710: 181 masked + 3 background = 184 selected slices
P2728: 177 maske

Epoch 1/10: 100%|██████████| 20965/20965 [18:38<00:00, 18.75it/s, D=0.6032, G=5.2408, L1=0.0454]  



Epoch 1 completed
Generator loss: 5.478421058976477
Discriminator loss: 0.6178376013581656
L1 loss: 0.044265558328936666
Adversarial loss: 1.0518652189902795

Running full-volume validation...

Validating: P2658
Full MAE: 0.086158 | Masked MAE: 0.157861 | SSIM: 0.565288 | PSNR: 15.7702

Validating: P6123
Full MAE: 0.094090 | Masked MAE: 0.151829 | SSIM: 0.514198 | PSNR: 15.3829

Validating: P7363
Full MAE: 0.086705 | Masked MAE: 0.167391 | SSIM: 0.524102 | PSNR: 15.7131

Validating: P7480
Full MAE: 0.078641 | Masked MAE: 0.152437 | SSIM: 0.590961 | PSNR: 16.3176

Validating: P8947
Full MAE: 0.080753 | Masked MAE: 0.154255 | SSIM: 0.568976 | PSNR: 16.1219

Validation averages:
full_volume_mae      0.085269
masked_mae           0.156755
mean_slice_ssim      0.552705
full_volume_psnr    15.861148
dtype: float64

New best generator checkpoint saved!

Epoch 1 finished.


Epoch 2/10: 100%|██████████| 20965/20965 [18:56<00:00, 18.45it/s, D=0.5913, G=4.2639, L1=0.0353] 



Epoch 2 completed
Generator loss: 4.702791813277499
Discriminator loss: 0.6366292862186504
L1 loss: 0.03756361042905804
Adversarial loss: 0.9464307647085605

Running full-volume validation...

Validating: P2658
Full MAE: 0.075649 | Masked MAE: 0.136485 | SSIM: 0.540912 | PSNR: 15.9868

Validating: P6123
Full MAE: 0.085790 | Masked MAE: 0.137714 | SSIM: 0.500533 | PSNR: 15.4474

Validating: P7363
Full MAE: 0.074603 | Masked MAE: 0.142926 | SSIM: 0.480513 | PSNR: 16.2390

Validating: P7480
Full MAE: 0.068793 | Masked MAE: 0.128948 | SSIM: 0.570920 | PSNR: 16.5138

Validating: P8947
Full MAE: 0.069496 | Masked MAE: 0.130654 | SSIM: 0.536786 | PSNR: 16.6198

Validation averages:
full_volume_mae      0.074866
masked_mae           0.135345
mean_slice_ssim      0.525933
full_volume_psnr    16.161359
dtype: float64

New best generator checkpoint saved!

Epoch 2 finished.


Epoch 3/10: 100%|██████████| 20965/20965 [34:05<00:00, 10.25it/s, D=0.5467, G=6.7211, L1=0.0567]  


... (11 outputs left out)

Epoch 9/10: 100%|██████████| 20965/20965 [17:37<00:00, 19.82it/s, D=0.7349, G=0.7220, L1=0.0001] 



Epoch 9 completed
Generator loss: 5.639427774986787
Discriminator loss: 0.5211910724334264
L1 loss: 0.04118293749315504
Adversarial loss: 1.5211340205789665

Running full-volume validation...

Validating: P2658
Full MAE: 0.073519 | Masked MAE: 0.133159 | SSIM: 0.525616 | PSNR: 16.1709

Validating: P6123
Full MAE: 0.085587 | Masked MAE: 0.137355 | SSIM: 0.479353 | PSNR: 15.4387

Validating: P7363
Full MAE: 0.069934 | Masked MAE: 0.133590 | SSIM: 0.473526 | PSNR: 16.6600

Validating: P7480
Full MAE: 0.068253 | Masked MAE: 0.129274 | SSIM: 0.549331 | PSNR: 16.6937

Validating: P8947
Full MAE: 0.068552 | Masked MAE: 0.129250 | SSIM: 0.523232 | PSNR: 16.6305

Validation averages:
full_volume_mae      0.073169
masked_mae           0.132526
mean_slice_ssim      0.510212
full_volume_psnr    16.318755
dtype: float64

Epoch 9 finished.


Epoch 10/10: 100%|██████████| 20965/20965 [17:41<00:00, 19.75it/s, D=0.3838, G=8.2001, L1=0.0644] 



Epoch 10 completed
Generator loss: 5.779878352999318
Discriminator loss: 0.5040096014523077
L1 loss: 0.041643467613110566
Adversarial loss: 1.6155315867485023

Running full-volume validation...

Validating: P2658
Full MAE: 0.066850 | Masked MAE: 0.119731 | SSIM: 0.635856 | PSNR: 17.4001

Validating: P6123
Full MAE: 0.073433 | Masked MAE: 0.118100 | SSIM: 0.613354 | PSNR: 16.9692

Validating: P7363
Full MAE: 0.067340 | Masked MAE: 0.129347 | SSIM: 0.600923 | PSNR: 17.3744

Validating: P7480
Full MAE: 0.064035 | Masked MAE: 0.120785 | SSIM: 0.653814 | PSNR: 17.4878

Validating: P8947
Full MAE: 0.060483 | Masked MAE: 0.113756 | SSIM: 0.661650 | PSNR: 18.0082

Validation averages:
full_volume_mae      0.066428
masked_mae           0.120344
mean_slice_ssim      0.633120
full_volume_psnr    17.447952
dtype: float64

Epoch 10 finished.

PIX2PIX TRAINING COMPLETED
Best validation masked MAE: 0.11620762050151826
Latest checkpoint: D:\PS07_CT2MRI\models\pix2pix
Training history: D:\PS07_CT2MRI\